# ⚾ 棒球的空氣動力學：旋轉、縫線與球路
**Physics Lab · SPIN 投球旋轉實驗室的延伸筆記本**　[回到網站](https://arpowchao.github.io/apps/pitch-spin/)

這本筆記本分成三部分，從上往下依序執行（`執行階段 → 全部執行`，約 3–5 分鐘）：

1. **投球軌跡模型**：重力＋空氣阻力＋馬格努斯力，算出不同球種的位移（和網站上的 3D 版相同）。
2. **動手探究**：改轉速、轉軸、有效旋轉，畫出關係圖。
3. **2D 數值風洞**：用格子玻爾茲曼法（LBM）算出氣流，看見渦街、馬格努斯效應與縫線讓尾流偏移；解析度比網站版高。

> 圖上的文字用英文，避免 Colab 缺中文字型時出現方塊；說明都寫在文字格裡。

In [ ]:
import os, time, numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
QUICK = os.environ.get('NB_QUICK') == '1'   # 測試用：縮短計算
print('NumPy', np.__version__)

## 1. 投球軌跡模型

座標以**捕手視角**：$x$ 往一壘側、$y$ 由本壘指向投手丘、$z$ 往上。球受三個力：

$$m\vec a = m\vec g \;-\; \tfrac12\rho A C_D\, v\,\vec v \;+\; \tfrac12\rho A C_L\, v^2\,\hat{n},\qquad \hat n \parallel \vec\omega_\perp\times\vec v$$

- $C_D = 0.33$；升力係數 $C_L = 1/(2.32+0.4/S)$，$S = r\,\omega_\perp/v$（Sawicki, Hubbard & Stronge, 2003）。
- $\vec\omega_\perp$ 是和前進方向**垂直**的旋轉（有效旋轉）；沿著前進方向的「陀螺旋轉」不產生馬格努斯力。
- **轉軸時鐘方向**：從投手背後看，旋轉把球推向時針指的方向（12:00 往上＝純後旋，6:00 往下＝純上旋）。

In [ ]:
G, RHO, CD = 9.80665, 1.2, 0.33
M, R = 0.145, 0.0366
AREA = np.pi * R**2
PLATE_Y = 17 / 12 * 0.3048

def CL(S):
    return 1 / (2.32 + 0.4 / S) if S > 1e-9 else 0.0

def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def spin_vector(vhat, rpm, tilt, eff, hand='R'):
    w = rpm * 2 * np.pi / 60
    th = np.radians(tilt * 30)
    m = np.array([-np.sin(th), 0, np.cos(th)])          # pitcher's right (3B side) = -x
    m = unit(m - m.dot(vhat) * vhat)
    axisT = unit(np.cross(vhat, m))
    gyro = 1 if hand == 'R' else -1
    return w * eff * axisT + w * np.sqrt(max(0, 1 - eff**2)) * gyro * vhat

def accel(v, omega, magnus=True):
    s = np.linalg.norm(v); k = 0.5 * RHO * AREA / M
    a = np.array([0, 0, -G]) - k * CD * s * v
    if magnus and s > 0:
        vh = v / s
        wp = omega - omega.dot(vh) * vh
        if np.linalg.norm(wp) > 0:
            a = a + k * CL(R * np.linalg.norm(wp) / s) * s * s * unit(np.cross(wp, v))
    return a

def fly(p, v, omega, magnus=True, dt=5e-4):
    path, t = [p.copy()], 0.0
    while True:
        k1v = accel(v, omega, magnus); k1p = v
        k2v = accel(v + k1v * dt / 2, omega, magnus); k2p = v + k1v * dt / 2
        k3v = accel(v + k2v * dt / 2, omega, magnus); k3p = v + k2v * dt / 2
        k4v = accel(v + k3v * dt, omega, magnus); k4p = v + k3v * dt
        p2 = p + (k1p + 2 * k2p + 2 * k3p + k4p) * dt / 6
        v2 = v + (k1v + 2 * k2v + 2 * k3v + k4v) * dt / 6
        if p2[1] <= PLATE_Y:
            f = (p[1] - PLATE_Y) / (p[1] - p2[1])
            pe = p + (p2 - p) * f
            path.append(pe)
            return pe, v + (v2 - v) * f, t + dt * f, np.array(path)
        p, v, t = p2, v2, t + dt
        path.append(p.copy())

def simulate(speed_kmh=152, rpm=2300, tilt=1.0, eff=0.8, hand='R', target=(0.0, 0.76)):
    """Aim so the spinning ball crosses the zone at `target`; compare with the same release without spin."""
    p0 = np.array([-0.55 if hand == 'R' else 0.55, 16.7, 1.78])
    speed = speed_kmh / 3.6; D = p0[1] - PLATE_Y
    yaw = np.arctan2(target[0] - p0[0], D); pitch = np.arctan2(target[1] - p0[2], D)
    for _ in range(40):
        v0 = speed * np.array([np.cos(pitch) * np.sin(yaw), -np.cos(pitch) * np.cos(yaw), np.sin(pitch)])
        om = spin_vector(unit(v0), rpm, tilt, eff, hand)
        end = fly(p0, v0, om, True, 1e-3)[0]
        ex, ez = target[0] - end[0], target[1] - end[2]
        if abs(ex) < 1e-5 and abs(ez) < 1e-5: break
        yaw += ex / D; pitch += ez / D
    real = fly(p0, v0, om, True); ghost = fly(p0, v0, om, False)
    return dict(move_cm=(real[0] - ghost[0])[[0, 2]] * 100, time=real[2], plate_kmh=np.linalg.norm(real[1]) * 3.6,
                path=real[3], ghost=ghost[3], omega=om, v0=v0)

PRESETS = {  # name: (km/h, rpm, tilt, active-spin fraction, approx. MLB average movement cm (catcher view))
    'Four-seam':  (152, 2300, 1.00, 0.80, (-20, 41)),
    'Sinker':     (150, 2150, 2.00, 0.70, (-38, 20)),
    'Cutter':     (143, 2400, 11.75, 0.25, (6, 20)),
    'Slider':     (138, 2450, 9.50, 0.15, (13, 3)),
    'Sweeper':    (132, 2600, 9.00, 0.40, (38, 3)),
    'Curveball':  (128, 2550, 7.25, 0.36, (23, -25)),
    'Changeup':   (138, 1750, 2.25, 0.68, (-36, 15)),
    'Splitter':   (139, 1350, 2.50, 0.52, (-25, 8)),
}
rows = []
for name, (kmh, rpm, tilt, eff, obs) in PRESETS.items():
    r = simulate(kmh, rpm, tilt, eff)
    rows.append((name, kmh, rpm, tilt, eff, *r['move_cm'], r['time'], obs))
print(f"{'pitch':10s} {'km/h':>5s} {'rpm':>5s} {'tilt':>5s} {'eff':>4s} | {'HB cm':>6s} {'IVB cm':>6s} {'t (s)':>6s} | MLB avg (HB, IVB)")
for n, kmh, rpm, tilt, eff, hb, ivb, t, obs in rows:
    print(f"{n:10s} {kmh:5d} {rpm:5d} {tilt:5.2f} {eff:4.2f} | {hb:6.1f} {ivb:6.1f} {t:6.3f} | {obs}")

**位移圖（捕手視角）**：實心點是模型、空心圈是大聯盟右投手各球種的約略平均。右投手的手臂側是三壘側（圖的左邊）。

模型和實際最大的差距常出現在伸卡球、變速球：真實球路還有「縫線偏移尾流（SSW）」造成的額外位移，只算旋轉的模型抓不到。

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.2))
for n, kmh, rpm, tilt, eff, hb, ivb, t, obs in rows:
    ax.plot(hb, ivb, 'o', ms=9); ax.annotate(n, (hb, ivb), xytext=(6, 4), textcoords='offset points', fontsize=9)
    ax.plot(*obs, 'o', mfc='none', mec='#c4472f', ms=10)
    ax.plot([hb, obs[0]], [ivb, obs[1]], ':', color='gray', lw=.8)
ax.axhline(0, color='k', lw=.8); ax.axvline(0, color='k', lw=.8)
ax.set_xlim(-60, 60); ax.set_ylim(-60, 60); ax.set_aspect('equal')
ax.set_xlabel('horizontal break (cm)   ← 3B side (RHP arm side) | 1B side →')
ax.set_ylabel('induced vertical break (cm)')
ax.set_title('Spin-based movement (filled) vs approx. MLB averages (open)')
plt.show()

In [ ]:
# 同一個出手點與瞄準點，比較三種球的路徑（側面與捕手視角）
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={'width_ratios': [2, 1]})
for name in ['Four-seam', 'Curveball', 'Sweeper']:
    kmh, rpm, tilt, eff, _ = PRESETS[name]
    r = simulate(kmh, rpm, tilt, eff)
    a1.plot(r['path'][:, 1], r['path'][:, 2], label=name)
    a1.plot(r['ghost'][:, 1], r['ghost'][:, 2], '--', lw=.8, color=a1.lines[-1].get_color(), alpha=.6)
    a2.plot(r['path'][:, 0] * 100, r['path'][:, 2] * 100, label=name)
    a2.plot(r['ghost'][-1, 0] * 100, r['ghost'][-1, 2] * 100, 'x', color=a2.lines[-1].get_color())
a1.invert_xaxis(); a1.set_xlabel('distance from home plate y (m)'); a1.set_ylabel('height z (m)')
a1.set_title('Side view (dashed = same release, no spin)'); a1.legend()
a2.add_patch(plt.Rectangle((-25.3, 46), 50.6, 61, fill=False, lw=1.5))
a2.set_xlabel('x (cm)  catcher view'); a2.set_ylabel('z (cm)'); a2.set_aspect('equal')
a2.set_title('At the plate (x = no spin)')
plt.tight_layout(); plt.show()

### 互動：自己調一顆球
拖曳滑桿（Colab 會顯示滑桿；若沒出現，直接改 `simulate(...)` 的參數重新執行）。

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider
    def explore(speed_kmh=150, rpm=2300, tilt=1.0, efficiency=0.8):
        r = simulate(speed_kmh, rpm, tilt, efficiency)
        hb, ivb = r['move_cm']
        fig, ax = plt.subplots(figsize=(4.6, 4.6))
        ax.arrow(0, 0, hb, ivb, width=.8, color='#ef7a2c', length_includes_head=True)
        ax.set_xlim(-70, 70); ax.set_ylim(-70, 70); ax.set_aspect('equal')
        ax.axhline(0, color='k', lw=.8); ax.axvline(0, color='k', lw=.8)
        ax.set_title(f'HB {hb:.0f} cm, IVB {ivb:+.0f} cm, t = {r["time"]:.3f} s')
        plt.show()
    interact(explore, speed_kmh=IntSlider(value=150, min=110, max=165),
             rpm=IntSlider(value=2300, min=0, max=3200, step=50),
             tilt=FloatSlider(value=1.0, min=0, max=11.75, step=0.25),
             efficiency=FloatSlider(value=0.8, min=0, max=1, step=0.05))
except ImportError:
    print('ipywidgets not available — edit the parameters of simulate(...) instead.')

## 2. 動手探究

**探究 A**：固定轉速 2300 rpm、轉軸 12:00，有效旋轉從 0 到 100%，垂直位移怎麼變？
**探究 B**：固定有效旋轉，轉速從 1000 到 3000 rpm，位移和轉速成正比嗎？（提示：$C_L$ 不是線性的。）

In [ ]:
effs = np.linspace(0, 1, 11)
ivb_e = [simulate(150, 2300, 12, e)['move_cm'][1] for e in effs]
rpms = np.arange(1000, 3001, 250)
ivb_r = [simulate(150, w, 12, 0.9)['move_cm'][1] for w in rpms]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(effs * 100, ivb_e, 'o-'); a1.set_xlabel('active spin (%)'); a1.set_ylabel('IVB (cm)'); a1.set_title('A: 2300 rpm, 12:00')
a2.plot(rpms, ivb_r, 'o-', label='model')
a2.plot(rpms, ivb_r[-1] * rpms / rpms[-1], '--', label='proportional to rpm')
a2.set_xlabel('spin rate (rpm)'); a2.set_ylabel('IVB (cm)'); a2.set_title('B: 90 % active, 12:00'); a2.legend()
plt.tight_layout(); plt.show()

## 3. 2D 數值風洞：看見氣流

真正的棒球在 $Re\approx10^5$，需要 3D、紊流的模擬或風洞。這裡和網站上的「棒球氣流實驗室」一樣，用 **2D 格子玻爾茲曼法（D2Q9、TRT 碰撞）** 在低雷諾數下算一個圓形截面，但格點更細（球徑 32 格）。用 Numba 編譯，Colab 上每個情況約 30–60 秒。

- 旋轉以表面切向速度設定，$S = \omega R / V$；正值＝畫面上逆時針。
- 球往左飛時，**順時針＝後旋**（升力向上）、逆時針＝上旋。
- 縫線是固定在表面的小凸起（放大 6 倍，才看得到）。
- 升力、阻力由每條邊界連結的動量交換加總，只對**完整的渦旋週期**取平均。

In [ ]:
import numpy as np
try:
    from numba import njit
except ImportError:  # very slow fallback
    print("Numba not found: pip install numba for speed")
    def njit(*a, **k):
        return (lambda f: f) if not (a and callable(a[0])) else a[0]

CX = np.array([0, 1, 0, -1, 0, 1, -1, -1, 1], dtype=np.int64)
CY = np.array([0, 0, 1, 0, -1, 1, 1, -1, -1], dtype=np.int64)
OPP = np.array([0, 3, 4, 1, 2, 7, 8, 5, 6], dtype=np.int64)
MIR = np.array([0, 1, 4, 3, 2, 8, 7, 6, 5], dtype=np.int64)
W = np.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36])


@njit(cache=True)
def _feq(q, d, ux, uy):
    cu = CX[q] * ux + CY[q] * uy
    return W[q] * d * (1 + 3 * cu + 4.5 * cu * cu - 1.5 * (ux * ux + uy * uy))


@njit(cache=True)
def _run(f, fn, solid, corr, sponge, eqfree, rho, ux, uy, U, wp, wm, nsteps, fxh, fyh, k0):
    # f, fn: (ny, nx, 9) populations; corr: (ny, nx, 9) moving-wall terms
    ny, nx = solid.shape
    post = np.empty(9)
    fl = np.empty(9)
    ns = sponge.shape[0]
    for it in range(nsteps):
        fx = 0.0
        fy = 0.0
        for y in range(ny):
            for x in range(nx):
                if solid[y, x]:
                    continue
                if x == 0:
                    d = rho[y, 1]
                    vx = U
                    vy = 0.0
                    for q in range(9):
                        fl[q] = _feq(q, d, vx, vy)
                else:
                    d = 0.0
                    for q in range(9):
                        fl[q] = f[y, x, q]
                        d += fl[q]
                    vx = (fl[1] - fl[3] + fl[5] - fl[6] - fl[7] + fl[8]) / d
                    vy = (fl[2] - fl[4] + fl[5] + fl[6] - fl[7] - fl[8]) / d
                rho[y, x] = d
                ux[y, x] = vx
                uy[y, x] = vy
                u2 = 1.5 * (vx * vx + vy * vy)
                post[0] = fl[0] - wp * (fl[0] - 4.0 / 9.0 * d * (1 - u2))
                for k in range(4):
                    if k == 0:
                        i, j, w, cu = 1, 3, 1.0 / 9, vx
                    elif k == 1:
                        i, j, w, cu = 2, 4, 1.0 / 9, vy
                    elif k == 2:
                        i, j, w, cu = 5, 7, 1.0 / 36, vx + vy
                    else:
                        i, j, w, cu = 6, 8, 1.0 / 36, -vx + vy
                    ep = w * d * (1 + 4.5 * cu * cu - u2)
                    em = w * d * 3 * cu
                    fp = 0.5 * (fl[i] + fl[j])
                    fm = 0.5 * (fl[i] - fl[j])
                    post[i] = fl[i] - wp * (fp - ep) - wm * (fm - em)
                    post[j] = fl[j] - wp * (fp - ep) + wm * (fm - em)
                if x < ns:
                    for q in range(9):
                        post[q] += sponge[x] * (eqfree[q] - post[q])
                fn[y, x, 0] = post[0]
                for q in range(1, 9):
                    tx = x + CX[q]
                    if tx < 0 or tx >= nx:
                        continue
                    ty = y + CY[q]
                    tq = q
                    if ty < 0 or ty >= ny:
                        ty = y
                        tq = MIR[q]
                    if solid[ty, tx]:
                        back = post[q] + d * corr[y, x, q]
                        fn[y, x, OPP[q]] = back
                        fx += CX[q] * (post[q] + back)
                        fy += CY[q] * (post[q] + back)
                    else:
                        fn[ty, tx, tq] = post[q]
        for y in range(ny):
            for q in range(9):
                fn[y, nx - 1, q] = fn[y, nx - 2, q]
        tmp = f
        f = fn
        fn = tmp
        fxh[k0 + it] = fx
        fyh[k0 + it] = fy
    return f, fn


class Tunnel:
    """2D TRT lattice-Boltzmann flow past a (spinning, optionally seamed) circle — same model as the website.
    y index grows downward; positive spin_ratio = counter-clockwise on screen (top surface moves upstream = topspin
    for a ball flying to the left, i.e. lift pushes DOWN)."""

    def __init__(self, D=32, Re=135, spin_ratio=0.0, seams=(), seam_height=0.0, U=0.1, kick=True, length=13.5, height=6.7):
        self.D, self.U, self.Re = D, U, Re
        self.nx = int(length * D); self.ny = int(height * D) | 1
        nx, ny = self.nx, self.ny
        self.cx, self.cy, R = int(3.5 * D), (ny - 1) / 2, D / 2
        nu = U * D / Re
        self.tau = 0.5 + 3 * nu
        self.wp, self.wm = 1 / self.tau, 1 / (0.5 + (3 / 16) / (self.tau - 0.5))
        om = spin_ratio * U / R
        Y, X = np.mgrid[0:ny, 0:nx].astype(float)
        dx, dy = X - self.cx, Y - self.cy
        solid = dx * dx + dy * dy <= R * R
        self.seam_centers = []
        if seam_height > 0:
            sr = max(0.75, 1.35 * seam_height); radial = R + seam_height - sr
            for ang in seams:
                a = np.radians(ang)
                sx, sy = self.cx + radial * np.cos(a), self.cy - radial * np.sin(a)
                solid |= (X - sx) ** 2 + (Y - sy) ** 2 <= sr * sr
                self.seam_centers.append((sx, sy, sr))
        self.solid = solid
        Xu, Yu = dx, -dy
        r2 = np.maximum(Xu * Xu + Yu * Yu, 1e-9)
        ux = U * (1 - R * R * (Xu * Xu - Yu * Yu) / r2 ** 2)
        uy = U * R * R * 2 * Xu * Yu / r2 ** 2
        if kick:
            kx, ky = X - (self.cx + 1.1 * D), Y - (self.cy - 0.3 * D)
            kr2 = np.maximum(kx * kx + ky * ky, 1e-9); rc2 = (0.35 * D) ** 2
            sw = 0.6 * U * np.sqrt(rc2) * (1 - np.exp(-kr2 / rc2)) / kr2
            ux, uy = ux - sw * ky, uy + sw * kx
        ux = np.where(solid, om * dy, ux); uy = np.where(solid, -om * dx, uy)
        rho = np.where(solid, 1.0, 1 + 1.5 * (U * U - ux * ux - uy * uy))
        cu = CX[:, None, None] * ux + CY[:, None, None] * uy
        self.f = np.ascontiguousarray(np.moveaxis(W[:, None, None] * rho * (1 + 3 * cu + 4.5 * cu * cu - 1.5 * (ux * ux + uy * uy)), 0, -1))
        self.fn = np.zeros_like(self.f)
        self.rho, self.ux, self.uy = rho.copy(), ux.copy(), uy.copy()
        corr = np.zeros((9, ny, nx))
        for q in range(1, 9):
            nb = np.zeros_like(solid)
            ys, xs = np.nonzero(~solid)
            tx, ty = xs + CX[q], ys + CY[q]
            ok = (tx >= 0) & (tx < nx) & (ty >= 0) & (ty < ny)
            ys, xs, tx, ty = ys[ok], xs[ok], tx[ok], ty[ok]
            hit = solid[ty, tx]
            ys, xs = ys[hit], xs[hit]
            mx, my = xs + 0.5 * CX[q], ys + 0.5 * CY[q]
            ddx, ddy = mx - self.cx, my - self.cy
            nxv, nyv, dist = ddx.copy(), ddy.copy(), np.hypot(ddx, ddy) - R
            for (sx, sy, sr) in self.seam_centers:
                sdx, sdy = mx - sx, my - sy; sd = np.hypot(sdx, sdy) - sr; c = sd < dist
                dist = np.where(c, sd, dist); nxv = np.where(c, sdx, nxv); nyv = np.where(c, sdy, nyv)
            nl = np.maximum(np.hypot(nxv, nyv), 1e-12); nxv, nyv = nxv / nl, nyv / nl
            wx, wy = om * ddy, -om * ddx
            vn = wx * nxv + wy * nyv
            corr[q, ys, xs] = -6 * W[q] * (CX[q] * (wx - vn * nxv) + CY[q] * (wy - vn * nyv))
        self.corr = np.ascontiguousarray(np.moveaxis(corr, 0, -1))
        n = int(0.75 * D)
        self.sponge = 0.3 * ((n - np.arange(n)) / n) ** 2
        self.eqfree = np.array([W[q] * (1 + 3 * CX[q] * U + 4.5 * (CX[q] * U) ** 2 - 1.5 * U * U) for q in range(9)])
        self.solid_u8 = solid.astype(np.uint8)
        self.fxh = np.zeros(0); self.fyh = np.zeros(0)
        self.steps = 0

    def step(self, n):
        k0 = self.steps
        self.fxh = np.concatenate([self.fxh, np.zeros(n)]); self.fyh = np.concatenate([self.fyh, np.zeros(n)])
        self.f, self.fn = _run(self.f, self.fn, self.solid_u8, self.corr, self.sponge, self.eqfree,
                               self.rho, self.ux, self.uy, self.U, self.wp, self.wm, n, self.fxh, self.fyh, k0)
        self.steps += n

    def passes(self):
        return self.steps * self.U / self.D

    def coefficients(self):
        q = 0.5 * self.U ** 2 * self.D
        return self.fxh / q, -self.fyh / q          # C_D(t), C_L(t), C_L > 0 = up

    def vorticity(self):
        w = np.gradient(self.uy, axis=1) - np.gradient(self.ux, axis=0)
        return np.where(self.solid, np.nan, -w * self.D / self.U)   # sign flipped: + = counter-clockwise as seen


def cycle_mean(cl, cd, U, D, warm_passes=4):
    """Average over whole shedding cycles after the warm-up (upward crossings of the smoothed lift)."""
    start = int(warm_passes * D / U)
    c = np.convolve(cl, np.ones(80) / 80, mode='same')[start:]
    x = c - c.mean()
    up = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]
    if len(up) >= 3:
        a, b = up[1] + start, up[-1] + start
        return cl[a:b].mean(), cd[a:b].mean(), len(up) - 2
    return cl[start:].mean(), cd[start:].mean(), 0

In [ ]:
D = 24 if QUICK else 32
PASSES = 10 if QUICK else 32            # 氣流流過幾個球徑
seam_h = 1.5 * 6 * D / 73                # 1.5 mm 縫線放大 6 倍（以格點計）
CASES = {
    'smooth, no spin':          dict(),
    'backspin S = -0.19':       dict(spin_ratio=-0.191),
    'topspin S = +0.19':        dict(spin_ratio=0.191),
    'one seam at 95°, no spin': dict(seams=(95,), seam_height=seam_h),
}
results, frames = {}, {}
for name, kw in CASES.items():
    t0 = time.time()
    tun = Tunnel(D=D, Re=135, **kw)
    steps = int(PASSES * D / tun.U)
    snap_every = int(0.25 * D / tun.U)
    frames[name] = []
    done = 0
    while done < steps:
        n = min(snap_every, steps - done); tun.step(n); done += n
        if tun.passes() > PASSES - 8:
            frames[name].append(tun.vorticity().copy())
    cd, cl = tun.coefficients()
    results[name] = dict(tunnel=tun, cd=cd, cl=cl, mean=cycle_mean(cl, cd, tun.U, tun.D))
    m = results[name]['mean']
    print(f'{name:26s}  mean C_L {m[0]:+.3f}  mean C_D {m[1]:.3f}  ({m[2]} full cycles)   {time.time() - t0:.0f} s')

**渦度圖**（紅＝逆時針、藍＝順時針）。光滑、不轉的球：渦旋上下交替脫落（卡門渦街），平均升力約 0。旋轉或單側縫線讓尾流偏向一邊，球受到相反方向的側向力。

> 後旋與上旋的升力大小不會剛好相反（例如 +0.44 與 −0.53）：起始擾動只放在尾流的一側，而且只平均了幾個完整週期。把 `PASSES` 調大，兩者會更接近。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 5.6))
for ax, (name, res) in zip(axes.flat, results.items()):
    tun = res['tunnel']
    ax.imshow(frames[name][-1], cmap='RdBu_r', vmin=-10, vmax=10, origin='upper')
    ax.add_patch(plt.Circle((tun.cx, tun.cy), tun.D / 2, color='#f3eee1'))
    for sx, sy, sr in tun.seam_centers:
        ax.add_patch(plt.Circle((sx, sy), sr, color='#c4162b'))
    cl = res['mean'][0]
    if abs(cl) > 0.1:   # draw the mean side force only when it clearly differs from zero
        ax.annotate('', xy=(tun.cx, tun.cy - np.sign(cl) * tun.D * (0.6 + abs(cl))), xytext=(tun.cx, tun.cy),
                    arrowprops=dict(color='gold', width=2.5, headwidth=8))
    ax.set_title(f'{name}   mean C_L = {cl:+.2f}'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
for name, res in results.items():
    tun = res['tunnel']
    t = np.arange(len(res['cl'])) * tun.U / tun.D
    smooth = np.convolve(res['cl'], np.ones(80) / 80, mode='same')
    ax.plot(t, smooth, label=name)
ax.axhline(0, color='k', lw=.8); ax.set_xlim(2, None)
ax.set_xlabel('time (flow-through diameters, tU/D)'); ax.set_ylabel('lift coefficient C_L (+ = up)')
ax.legend(fontsize=8, ncol=2); ax.set_title('Lift oscillates with vortex shedding; spin and a seam shift its mean')
plt.show()

In [ ]:
# 動畫：比較光滑球與單側縫線（約 8 個球徑）
names = ['smooth, no spin', 'one seam at 95°, no spin']
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
ims = []
for ax, name in zip(axes, names):
    tun = results[name]['tunnel']
    ims.append(ax.imshow(frames[name][0], cmap='RdBu_r', vmin=-10, vmax=10))
    ax.add_patch(plt.Circle((tun.cx, tun.cy), tun.D / 2, color='#f3eee1'))
    for sx, sy, sr in tun.seam_centers:
        ax.add_patch(plt.Circle((sx, sy), sr, color='#c4162b'))
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
n = min(len(frames[k]) for k in names)
def update(i):
    for im, name in zip(ims, names):
        im.set_data(frames[name][i])
    return ims
anim = animation.FuncAnimation(fig, update, frames=n, interval=120)
plt.close(fig)
HTML(anim.to_jshtml())

## 4. 為什麼不直接做 3D？

| | 這本筆記本（2D LBM） | 真實棒球 |
|---|---|---|
| 雷諾數 | 135（教學用） | 約 $1.5\times10^5$ |
| 邊界層 | 層流、粗格點 | 很薄，在縫線處轉捩成紊流 |
| 幾何 | 圓形截面＋放大的凸起 | 3D 球、108 針縫線、會旋轉 |
| 計算量 | 筆電／Colab 幾分鐘 | 3D LES 需數千萬格點、超級電腦或風洞 |

所以：**軌跡模型**（第 1、2 部分）用風洞量到的升力係數，能可靠地預測旋轉造成的位移；**2D 風洞**（第 3 部分）適合看見「為什麼」——旋轉與縫線如何讓尾流偏移、產生側向力——但數值不能直接當成真球的受力。

**參考**
- Sawicki, Hubbard & Stronge (2003). *About the physics of a curveball.* Am. J. Phys. 71, 1152.
- Alan Nathan, *The Physics of Baseball*: https://baseball.physics.illinois.edu/
- Smith & Smith (2021). *Seam shifted wake.* Proc. IMechE Part P.
- Ginzburg, Verhaeghe & d'Humières (2008). Two-relaxation-time lattice Boltzmann scheme. Commun. Comput. Phys. 3, 427.